# Эффективность на практике

In [ ]:
!pip install bitsandbytes transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 32.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
from huggingface_hub import interpreter_login
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import torch, time, random, numpy as np
from tqdm import tqdm
import gc
seed_value = 42
set_seed(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
interpreter_login()


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Enter your token (input will not be visible): ··········
Add token as git credential? (Y/n) Y


## KV-cache

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2").cuda()
tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
prompt = "Once upon a time,"

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids.cuda()
attention_mask = inputs.attention_mask.cuda()

# Параметры генерации с температурой
generation_kwargs = {
    "max_new_tokens": 50,
    "do_sample": True,
    "temperature": 0.7,
    "attention_mask": attention_mask,
    "pad_token_id": tokenizer.eos_token_id,
    "eos_token_id": None
}

# сколько раз будем запускать генерацию
n_runs = 10

times_no_cache = []
tokens_no_cache_count = []

# Генерация без использования кеша
for i in tqdm(range(n_runs), desc="Generating without cache"):
    start = time.time()
    outputs = model.generate(input_ids, use_cache=False, **generation_kwargs)
    elapsed = time.time() - start
    times_no_cache.append(elapsed)

times_cache = []
tokens_cache_count = []

# Генерация с использованием кеша
for i in tqdm(range(n_runs), desc="Generating with cache"):
    start = time.time()
    outputs = model.generate(input_ids, use_cache=True, **generation_kwargs)
    elapsed = time.time() - start
    times_cache.append(elapsed)

avg_time_no_cache = sum(times_no_cache) / n_runs
avg_time_cache = sum(times_cache) / n_runs
print()
print(f"Без кеша: Среднее время: {avg_time_no_cache:.2f} сек")
print(f"С кешем: Среднее время: {avg_time_cache:.2f} сек")
print(f"Ускорение x{avg_time_no_cache/avg_time_cache:.2f}")

Generating with cache: 100%|██████████| 10/10 [00:04<00:00,  2.06it/s]


Без кеша: Среднее время: 0.61 сек
С кешем: Среднее время: 0.48 сек
Ускорение x1.26


In [ ]:
del outputs
del input_ids
del attention_mask
del model

## Speculative decoding

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# IDs моделей
small_model_id = "gpt2" # 124M
large_model_id = "gpt2-xl" # 1.5b

# Загружаем модели
small_model = AutoModelForCausalLM.from_pretrained(small_model_id, device_map="cuda")
large_model = AutoModelForCausalLM.from_pretrained(large_model_id, device_map="cuda")

# Загружаем общий токенизатор
tokenizer = AutoTokenizer.from_pretrained(small_model_id)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
max_new_tokens = 15
N = 3 # Количество токенов, предсказанных маленькой моделью за шаг

def baseline_generation(prompt, max_new_tokens):
    # Токенизируем промпт
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids.cuda()
    generated = input_ids.clone()

    while (generated.shape[1] - input_ids.shape[1]) < max_new_tokens:
        attn_mask = torch.ones_like(generated)
        with torch.no_grad():
            out = large_model.generate(
                generated,
                max_new_tokens=1,
                do_sample=False, # жадная генерация
                attention_mask=attn_mask,
                pad_token_id=tokenizer.pad_token_id
            )
        new_token = out[:, -1:]
        generated = torch.cat((generated, new_token), dim=1)
    return generated

prompt = "President of United States of America is"
torch.cuda.synchronize()
start_time = time.time()
baseline_ids = baseline_generation(prompt, max_new_tokens)
torch.cuda.synchronize()
baseline_time = time.time() - start_time

baseline_text = tokenizer.decode(baseline_ids[0], skip_special_tokens=True)
print("\nBaseline output:")
print(baseline_text)
print("Baseline time: {:.2f} sec".format(baseline_time))

# спекулятивный декодинг

def get_large_preds_parallel(generated, candidate_tokens, N):
    """
    Для текущей последовательности `generated` и кандидатных токенов (от маленькой модели)
    формируем список префиксов:
      - prefix0 = generated
      - prefix_j = generated + candidate_tokens[:, :j]  для j=1,...,N
    и за один батч вычисляем предсказание следующего токена от большой модели для каждого префикса.
    Возвращаем список длины N+1, где:
      large_preds[0] соответствует префиксу с j=0,
      large_preds[j] для j>=1 соответствует префиксу с j токенами кандидата.
    """
    prefixes = []
    # j=0
    prefixes.append(generated)
    # j от 1 до N
    for j in range(1, N+1):
        prefixes.append(torch.cat((generated, candidate_tokens[:, :j]), dim=1))

    pad_token_id = tokenizer.pad_token_id
    # Определяем максимальную длину среди префиксов
    prefix_lens = [p.shape[1] for p in prefixes]
    max_len = max(prefix_lens)

    padded_prefixes = []
    attention_masks = []
    for p in prefixes:
        pad_length = max_len - p.shape[1]
        if pad_length > 0:
            padding = torch.full((p.shape[0], pad_length), pad_token_id, device=p.device)
            p_padded = torch.cat([p, padding], dim=1)
            mask = torch.cat([torch.ones(p.shape[1], device=p.device), torch.zeros(pad_length, device=p.device)])
        else:
            p_padded = p
            mask = torch.ones(p.shape[1], device=p.device)
        padded_prefixes.append(p_padded)
        attention_masks.append(mask)

    # Собираем батч
    batch = torch.cat(padded_prefixes, dim=0)  # размер батча = N+1, seq_len = max_len
    attention_mask = torch.stack(attention_masks, dim=0)

    with torch.no_grad():
        outputs = large_model(input_ids=batch, attention_mask=attention_mask)
    logits = outputs.logits  # shape: (N+1, max_len, vocab_size)

    preds = []
    for i, seq_len in enumerate(prefix_lens):
        last_logits = logits[i, seq_len - 1, :]  # логиты для последнего "настоящего" токена
        pred_token = torch.argmax(last_logits, dim=-1).item()
        preds.append(pred_token)

    return preds
def speculative_decoding(prompt, max_new_tokens, N):
    """
    Спекулятивный декодинг с параллельным получением предсказаний от большой модели.
    Алгоритм:
      1. Маленькая модель жадно генерирует N токенов c1, c2, ..., c_N.
      2. Параллельно большая модель жано генерирует следующие токены для префиксов L0, L1, ..., L_N:
             x1,...,x_i        -> L0
             x1,...,x_i, c1     -> L1
             x1,...,x_i, c1,c2   -> L2
             ...
             x1,...,x_i, c1,..., c_N -> L_N
      3. Сравниваем кандидатные токены c1...c_N с L1...L_N:
             m = наибольшее число подряд совпадающих токенов.
      4. Добавляем к последовательности: candidate_tokens[:, :m] и затем токен L_m (из большой модели).
      5. Повторяем, пока не наберём max_new_tokens.
    """
    # Токенизируем промпт
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids.cuda()
    # Создаем attention_mask для исходной последовательности
    generated = input_ids.clone()  # текущая последовательность x1,..., x_i

    while (generated.shape[1] - input_ids.shape[1]) < max_new_tokens:
        # 1. Маленькая модель жадно генерирует N токенов c1, c2, ..., c_N.
        attn_mask = torch.ones_like(generated)
        with torch.no_grad():
            candidate_out = small_model.generate(
                generated,
                max_new_tokens=N,
                do_sample=False, # жадная генерация
                attention_mask=attn_mask,
                pad_token_id=tokenizer.pad_token_id
            )
        candidate_tokens = candidate_out[:, -N:]  # размер [1, N]

        # 2. Параллельно вычисляем предсказания большой модели для N+1 префиксов
        # large_preds[0] соответствует префиксу с j=0,
        # large_preds[j] для j>=1 соответствует префиксу с j токенами кандидата.
        large_preds = get_large_preds_parallel(generated, candidate_tokens, N)

        # 3. Сравниваем токены-кандидаты с предсказаниями для j=1,...,N
        # находим m - количество первых совпавших токенов
        m = 0
        for k in range(N):
            # Сравниваем candidate token c_{k+1} с large_preds[k+1]
            if candidate_tokens[0, k].item() == large_preds[k+1]:
                m += 1
            else:
                break

        # 4. Формируем новые токены:
        #    Если m > 0, добавляем m токенов-кандидатов, иначе (m==0) ничего из кандидатов.
        #    Всегда добавляем один токен от большой модели – берем предсказание с индексом m.
        new_tokens_candidate = candidate_tokens[:, :m] if m > 0 else candidate_tokens[:, :0]
        extra_token = torch.tensor([[large_preds[m]]], device=generated.device)
        to_add = torch.cat((new_tokens_candidate, extra_token), dim=1)

        generated = torch.cat((generated, to_add), dim=1)

        # Если получилось больше токенов, чем нужно – обрезаем
        if (generated.shape[1] - input_ids.shape[1]) > max_new_tokens:
            generated = generated[:, :input_ids.shape[1] + max_new_tokens]
            break

    return generated



# Тестируем спекулятивный декодинг
torch.cuda.synchronize()
start_time = time.time()
speculative_ids = speculative_decoding(prompt, max_new_tokens, N)
torch.cuda.synchronize()
speculative_time = time.time() - start_time

speculative_text = tokenizer.decode(speculative_ids[0], skip_special_tokens=True)
print("\nSpeculative output:")
print(speculative_text)
print("Speculative time: {:.2f} sec".format(speculative_time))

# Сравним времена
print("\nBaseline time: {:.2f} sec".format(baseline_time))
print("Speculative time: {:.2f} sec".format(speculative_time))
print("Speedup: {:.2f}x".format(baseline_time / speculative_time))



Baseline output:
President of United States of America is the highest office in the land. It is the highest office in the land
Baseline time: 0.72 sec

Speculative output:
President of United States of America is the highest office in the land. It is the highest office in the land
Speculative time: 1.50 sec

Baseline time: 0.72 sec
Speculative time: 1.50 sec
Speedup: 0.48x


In [ ]:
del small_model
del large_model
del speculative_ids
del baseline_ids
gc.collect()
torch.cuda.empty_cache()